# Taux de change en XOF via l'API OANDA

Collecte automatique des taux journaliers **Bid** de MRU, GHS, USD, NGN et GNF exprimés en XOF, avec la clé API OANDA, et écriture dans `Cours devises` d'un fichier par devise et par mois (`USD_XOF Octobre 2026.xlsx`), au format exact des fichiers existants (cumul du 1er janvier à la fin du mois, ordre décroissant, lignes de synthèse).

Principes : clé jamais écrite dans le code, aucun fichier existant modifié sans sauvegarde, aucune valeur inventée, contrôles qualité avant écriture. Documentation : `docs/documentation_technique.md`.

In [1]:
# Installe automatiquement les packages manquants (première exécution)
import importlib.util, subprocess, sys
PACKAGES = {"requests": "requests", "pandas": "pandas", "openpyxl": "openpyxl", "matplotlib": "matplotlib", "keyring": "keyring"}
manquants = [pip for mod, pip in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if manquants:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *manquants])
    print("Installés :", ", ".join(manquants))
else:
    print("Tous les packages sont déjà installés.")

Tous les packages sont déjà installés.


## 1. Paramètres

In [2]:
import os, getpass, shutil
from datetime import date, timedelta
from pathlib import Path

DOSSIER = Path(os.environ.get("COURS_DEVISES_DIR", "../Cours devises"))
DEVISES = ["MRU", "GHS", "USD", "NGN", "GNF"]
CIBLE = "XOF"
MOIS_FR = ["Janvier", "Février", "Mars", "Avril", "Mai", "Juin", "Juillet",
           "Août", "Septembre", "Octobre", "Novembre", "Décembre"]

TELECHARGER = True        # False : ne rien télécharger
MOIS_CIBLE = "2026-04"         # None = mois précédent ; sinon "AAAA-MM" (ex. "2026-10")
FORCER = True            # True : régénérer même si le fichier existe (l'ancien est sauvegardé avant)
DATA_SET = None           # None = jeu OANDA par défaut ; "BCEAO" = jeu BCEAO (décision métier, voir documentation)
LANGUE_ENTETE = "EN"      # "EN" : Period Start Date / Period Average... ; "FR" : Date de début de la période / Moyenne...
ECRIRE_SI_INCOMPLET = False   # False : un mois avec jours manquants n'est PAS écrit (aucune valeur inventée)
SEUIL_SAUT = 0.15         # alerte si variation d'un jour à l'autre > 15 %
DOSSIER_SAUVEGARDE = DOSSIER / "_sauvegardes"
JOURNAL = DOSSIER / "_journal_collecte.csv"

ENTETES = {
    "EN": ["Daily BID rates @ +/- 0.0%", "OANDA Exchange Rates", "https://hcc.corporatefxservices.com",
           "Period Start Date", ["Period Average", "Period Low", "Period High"]],
    "FR": ["Taux SOUMISSION quotidiens @ +/- 0.0%", "Taux FX OANDA", "https://fxds-hcc.oanda.com",
           "Date de début de la période", ["Moyenne de la période", "Creux de la période", "Pic de la période"]],
}

def mois_vise():
    """Mois précédent (ou MOIS_CIBLE) : renvoie (1er janvier, dernier jour du mois)."""
    if MOIS_CIBLE:
        a, m = map(int, MOIS_CIBLE.split("-"))
        premier = date(a, m, 1)
    else:
        premier = (date.today().replace(day=1) - timedelta(days=1)).replace(day=1)
    fin = (premier.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
    return date(fin.year, 1, 1), fin

## 2. Client API OANDA
Clé lue dans `OANDA_FX_API_KEY`, sinon coffre Windows (`keyring`), sinon saisie masquée. Erreurs gérées : réseau, 401/403 (clé ou droits), 404, 429 (limite, attente et nouvel essai), 5xx, fenêtre de dates non autorisée, réponse vide. Le taux journalier Bid est le champ `average_bid` des chandeliers.

In [3]:
import requests, time, re
import pandas as pd
import keyring

API_URL = "https://api.corporatefxservices.com"


class ErreurAPI(RuntimeError):
    pass


class FenetreHorsPlage(ErreurAPI):
    def __init__(self, debut, fin):
        super().__init__(f"dates autorisées par la clé : {debut} au {fin}")
        self.debut, self.fin = debut, fin


def cle_api():
    cle = os.environ.get("OANDA_FX_API_KEY") or keyring.get_password("oanda_fx_api", "cle")
    if cle:
        return cle.strip()
    cle = getpass.getpass("Clé API OANDA : ").strip()
    if input("Mémoriser dans le coffre Windows ? (o/n) ").strip().lower().startswith("o"):
        keyring.set_password("oanda_fx_api", "cle", cle)
    return cle


def api_get(session, chemin, tentatives=4, **params):
    params = {k: v for k, v in params.items() if v is not None}
    for t in range(tentatives):
        try:
            r = session.get(f"{API_URL}{chemin}.json", params=params, timeout=30)
        except requests.RequestException as e:
            if t == tentatives - 1:
                raise ErreurAPI(f"connexion impossible : {e}")
            time.sleep(2 ** t)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            if t == tentatives - 1:
                raise ErreurAPI(f"HTTP {r.status_code} persistant (limite ou service indisponible)")
            time.sleep(int(r.headers.get("Retry-After", 2 ** (t + 1))))
            continue
        if r.status_code in (401, 403):
            raise ErreurAPI(f"HTTP {r.status_code} : clé refusée ou fonction non incluse dans l'abonnement ({r.text[:200]})")
        if r.status_code == 404:
            raise ErreurAPI(f"HTTP 404 : donnée ou paire indisponible ({r.text[:200]})")
        if not r.ok:
            m = re.search(r">= (\d{4}-\d{2}-\d{2}) and <= (\d{4}-\d{2}-\d{2})", r.text)
            if m:
                raise FenetreHorsPlage(m.group(1), m.group(2))
            raise ErreurAPI(f"HTTP {r.status_code} sur {chemin} : {r.text[:300]}")
        restant = r.headers.get("X-Rate-Limit-Remaining")
        if restant is not None:
            session.dernier_restant = restant
        return r.json()


def iso(jour, fin_de_journee=False):
    """Format exigé : AAAA-MM-JJThh:mm:ss+00:00, sur un quart d'heure UTC."""
    return f"{jour.isoformat()}T{'23:45' if fin_de_journee else '00:00'}:00+00:00"


def taux_bid(session, base, debut, fin):
    """Taux journaliers Bid (average_bid) de base/XOF, du plus récent au plus ancien. Lève ErreurAPI si vide."""
    data = api_get(session, "/v2/rates/candles", base=base, quote=CIBLE, fields="averages",
                   start_time=iso(debut), end_time=iso(fin, True), data_set=DATA_SET)
    quotes = data.get("quotes") or []
    if not quotes:
        raise ErreurAPI(f"aucune donnée renvoyée pour {base}/{CIBLE}")
    df = pd.DataFrame(quotes)
    df["date"] = pd.to_datetime(df["start_time"], utc=True).dt.tz_localize(None).dt.normalize()
    df["taux"] = pd.to_numeric(df["average_bid"], errors="coerce")
    df = df[(df["date"].dt.date >= debut) & (df["date"].dt.date <= fin)]
    return (df[["date", "taux"]].drop_duplicates("date").sort_values("date", ascending=False)
              .reset_index(drop=True))

## 3. Contrôles qualité et écriture Excel fidèle au modèle
Format reproduit : une feuille `rates_data`, 3 lignes d'en-tête, 2 lignes vides, ligne de titres, données dès la ligne 7 (date `mm-dd-yy`, taux en `General`), ligne vide, 3 lignes de synthèse (moyenne, creux, pic), police Aptos Narrow 11, colonnes de largeur 13. Taux à 6 chiffres significatifs, comme les fichiers existants. Un fichier existant n'est jamais écrasé sans copie préalable dans `_sauvegardes`.

In [4]:
from openpyxl import Workbook
from openpyxl.styles import Font
from datetime import datetime


def controles(df, debut, fin):
    """Renvoie (erreurs bloquantes, avertissements)."""
    err, warn = [], []
    attendus = pd.date_range(debut, fin, freq="D")
    if df["taux"].isna().any():
        err.append("valeurs vides ou non numériques")
    if (df["taux"] <= 0).any():
        err.append("taux nul ou négatif")
    manq = attendus.difference(df["date"])
    if len(manq):
        liste = ", ".join(d.strftime("%d/%m") for d in manq[:8]) + ("..." if len(manq) > 8 else "")
        (warn if ECRIRE_SI_INCOMPLET else err).append(f"{len(manq)} jour(s) manquant(s) sur {len(attendus)} : {liste}")
    s = df.sort_values("date")["taux"]
    sauts = s.pct_change().abs()
    if (sauts > SEUIL_SAUT).any():
        n = int((sauts > SEUIL_SAUT).sum())
        warn.append(f"{n} variation(s) journalière(s) > {SEUIL_SAUT:.0%} à vérifier")
    return err, warn


def sauvegarder(chemin):
    if chemin.exists():
        DOSSIER_SAUVEGARDE.mkdir(parents=True, exist_ok=True)
        copie = DOSSIER_SAUVEGARDE / f"{chemin.stem}_{datetime.now():%Y%m%d_%H%M%S}{chemin.suffix}"
        shutil.copy2(chemin, copie)
        return copie


def ecrire_fichier(df, base, fin, langue=None):
    """Écrit <BASE>_XOF <Mois> <Année>.xlsx au format des modèles. Sauvegarde l'existant avant remplacement."""
    t1, t2, t3, titre, synth = ENTETES[langue or LANGUE_ENTETE]
    df = df.sort_values("date", ascending=False)
    taux = [float(f"{x:.6g}") for x in df["taux"]]
    wb = Workbook()
    ws = wb.active
    ws.title = "rates_data"
    police = Font(name="Aptos Narrow", size=11)
    for ligne in ([t1, None], [t2, None], [t3, None], [None, None], [None, None], [titre, f"{base}/{CIBLE}"]):
        ws.append(ligne)
    for d, t in zip(df["date"], taux):
        ws.append([d.to_pydatetime(), t])
        ws.cell(ws.max_row, 1).number_format = "mm-dd-yy"
    ws.append([None, None])
    vals = [float(f"{sum(taux) / len(taux):.6g}"), min(taux), max(taux)]
    for lib, v in zip(synth, vals):
        ws.append([lib, v])
    for row in ws.iter_rows():
        for c in row:
            c.font = police
    ws.column_dimensions["A"].width = 13
    ws.column_dimensions["B"].width = 13
    chemin = DOSSIER / f"{base}_{CIBLE} {MOIS_FR[fin.month - 1]} {fin.year}.xlsx"
    DOSSIER.mkdir(parents=True, exist_ok=True)
    copie = sauvegarder(chemin)
    wb.save(chemin)
    return chemin, copie


def journaliser(ligne):
    ligne = {"horodatage": datetime.now().isoformat(timespec="seconds"), **ligne}
    pd.DataFrame([ligne]).to_csv(JOURNAL, mode="a", header=not JOURNAL.exists(), index=False, encoding="utf-8-sig")

## 4. Exécution (mois précédent, au 1er du mois n+1)
Contrôle de la date, détection des fichiers déjà présents (non retéléchargés), collecte, contrôles, écriture. Un échec sur une devise n'empêche pas les autres.

In [5]:
debut, fin = mois_vise()
nouveaux = {}   # seules les données téléchargées pendant cette exécution sont affichées
nom_mois = f"{MOIS_FR[fin.month - 1]} {fin.year}"
print(f"Aujourd'hui : {date.today()} | mois visé : {nom_mois} (du {debut} au {fin})")


def collecter(session=None):
    global nouveaux
    nouveaux = {}
    a_faire = []
    for d in DEVISES:
        existants = list(DOSSIER.glob(f"{d}_{CIBLE} {nom_mois}.*"))
        if existants and not FORCER:
            print("DÉJÀ PRÉSENT", existants[0].name)
        else:
            a_faire.append(d)
    if not a_faire:
        print("Rien à télécharger : le mois est complet dans le dossier.")
        return
    session = session or requests.Session()
    if "Authorization" not in session.headers:
        session.headers["Authorization"] = f"Bearer {cle_api()}"
    try:
        print("Quotes restants :", api_get(session, "/v2/remaining_quotes"))
    except ErreurAPI as e:
        print("ARRÊT :", e)
        return
    for d in a_faire:
        try:
            df = taux_bid(session, d, debut, fin)
            err, warn = controles(df, debut, fin)
            for w in warn:
                print(f"  ATTENTION {d} : {w}")
            if err:
                print(f"{d} -> NON ÉCRIT : " + " ; ".join(err))
                journaliser({"devise": d, "mois": nom_mois, "statut": "refusé", "detail": " ; ".join(err)})
                continue
            chemin, copie = ecrire_fichier(df, d, fin)
            nouveaux[d] = df.sort_values("date")
            print(d, "->", chemin.name, f"({len(df)} jours)" + (f", ancien fichier sauvegardé : {copie.name}" if copie else ""))
            journaliser({"devise": d, "mois": nom_mois, "statut": "écrit", "detail": f"{len(df)} jours; {' ; '.join(warn)}"})
        except FenetreHorsPlage as e:
            print(f"ARRÊT : l'API n'autorise que {e.debut} au {e.fin} avec cette clé ; période demandée {debut} au {fin}.")
            print("Aucun fichier écrit. Pour étendre la plage : webservices@corporatefxservices.com.")
            journaliser({"devise": d, "mois": nom_mois, "statut": "hors plage", "detail": f"{e.debut}..{e.fin}"})
            break
        except ErreurAPI as e:
            print(d, "-> ÉCHEC :", e)
            journaliser({"devise": d, "mois": nom_mois, "statut": "échec", "detail": str(e)[:200]})


if not TELECHARGER:
    print("Téléchargement désactivé (TELECHARGER = False).")
elif fin >= date.today():
    print("Impossible pour l'instant : le mois n'est pas terminé. Réessayez à partir du", fin + timedelta(days=1))
else:
    collecter()

Aujourd'hui : 2026-11-13 | mois visé : Avril 2026 (du 2026-01-01 au 2026-04-30)
Quotes restants : {'remaining_quotes': 'unlimited', 'used_quotes': 1046}
MRU -> ÉCHEC : HTTP 400 sur /v2/rates/candles : {"code":1,"message":"Invalid or malformed argument; [start_time] must be on the same hour and minute boundary as [end_time]: (2026-01-01T00:00:00+00:00, 2026-04-30T23:45:00+00:00)"}
GHS -> ÉCHEC : HTTP 400 sur /v2/rates/candles : {"code":1,"message":"Invalid or malformed argument; [start_time] must be on the same hour and minute boundary as [end_time]: (2026-01-01T00:00:00+00:00, 2026-04-30T23:45:00+00:00)"}
USD -> ÉCHEC : HTTP 400 sur /v2/rates/candles : {"code":1,"message":"Invalid or malformed argument; [start_time] must be on the same hour and minute boundary as [end_time]: (2026-01-01T00:00:00+00:00, 2026-04-30T23:45:00+00:00)"}
NGN -> ÉCHEC : HTTP 400 sur /v2/rates/candles : {"code":1,"message":"Invalid or malformed argument; [start_time] must be on the same hour and minute boundary

## 5. Outils complémentaires (optionnel, désactivé par défaut)
Fonctions de l'ancien notebook conservées : test de connexion, taux spot, contrats à terme supportés. Elles consomment des quotes ; activer `OUTILS = True` pour les utiliser.

In [6]:
OUTILS = False

def test_connexion(session):
    print("Quotes :", api_get(session, "/v2/remaining_quotes"))
    devs = api_get(session, "/v2/currencies")["currencies"]
    print(len(devs), "devises disponibles ; cibles présentes :", [c["code"] for c in devs if c["code"] in DEVISES + [CIBLE]])

def spot(session, bases=DEVISES):
    q = api_get(session, "/v2/rates/spot", base=bases, quote=CIBLE, data_set=DATA_SET)["quotes"]
    return pd.DataFrame(q)[["base_currency", "quote_currency", "date_time", "bid", "ask", "midpoint"]]

def forwards_supportes(session):
    return pd.DataFrame(api_get(session, "/v2/supported_forwards")["supported_forwards"])

if OUTILS:
    s = requests.Session(); s.headers["Authorization"] = f"Bearer {cle_api()}"
    test_connexion(s)
    display(spot(s))
    display(forwards_supportes(s).query("quote_currency == @CIBLE"))

## 6. Affichage des nouvelles données uniquement (matplotlib)

In [7]:
import matplotlib.pyplot as plt

ROUGE, GRIS = "#D0021B", "#9E9E9E"
if not nouveaux:
    print("Rien de nouveau à afficher : aucune donnée n'a été téléchargée pendant cette exécution.")
for d, s in nouveaux.items():
    fig, ax = plt.subplots(figsize=(10, 4), facecolor="white")
    ax.plot(s["date"], s["taux"], color=ROUGE, linewidth=1.4)
    ax.axhline(s["taux"].mean(), color=GRIS, linewidth=1, linestyle="--", label=f"moyenne {s['taux'].mean():.4f}")
    ax.set_title(f"{d}/{CIBLE} (Bid) : {s['date'].min():%d/%m/%Y} au {s['date'].max():%d/%m/%Y}")
    ax.set_xlabel("Date"); ax.set_ylabel(f"{CIBLE} pour 1 {d}")
    ax.grid(color=GRIS, alpha=0.3); ax.legend(); fig.tight_layout(); plt.show()

Rien de nouveau à afficher : aucune donnée n'a été téléchargée pendant cette exécution.
